# Summarizing Yesterday's Reddit Subreddit Posts

This project fetches yesterday's posts from a selected Reddit subreddit and uses OpenAI to generate a concise summary of the main topics, themes, and notable discussions.

## Step 1: Import libraries

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

## Step 2: Define a tool - Subreddit reader

In [ ]:
import requests
from datetime import datetime, timedelta, timezone

# This function retrieves posts published yesterday from a specified Reddit subreddit. It currently checks the latest 100 posts by default, but you can change the `limit` parameter to search through more or fewer posts.
def get_yesterday_reddit_posts(subreddit_name, limit=100):
    url = f"https://www.reddit.com/r/{subreddit_name}/new.json?limit={limit}"

    headers = {
        "User-Agent": "agentic-ai-lab-by-tatiana/0.1"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()

    now = datetime.now(timezone.utc)

    yesterday_start = datetime(
        year=now.year,
        month=now.month,
        day=now.day,
        tzinfo=timezone.utc
    ) - timedelta(days=1)

    yesterday_end = yesterday_start + timedelta(days=1)

    posts = []

    for item in data["data"]["children"]:
        post = item["data"]
        post_time = datetime.fromtimestamp(post["created_utc"], tz=timezone.utc)

        if yesterday_start <= post_time < yesterday_end:
            posts.append({
                "title": post.get("title", ""),
                "text": post.get("selftext", ""),
                "score": post.get("score", 0),
                "comments": post.get("num_comments", 0)
            })

    return posts

In [ ]:
# Get the posts from MicrosoftFabric subreddit from yesterday
posts_fabric = get_yesterday_reddit_posts("MicrosoftFabric", limit=100)

In [ ]:
# Display the posts retrieved
posts_fabric

[{'title': '95 GB data warehouse - Azure SQL DB in Fabric make sense.',
  'text': "We're migrating to Fabric from a current setup that is a SQL Server Data warehouse, largely fed by a Synapse Datalake containing Delta-Parquet, Parquet, and CSV files. \n\nFor the Datawarehouse part, given the relatively low data volume, sticking with a RDMS is appealing vs. Delta Parquet storage.  \n\nThat said, thoughts/ feedback?",
  'score': 8,
  'comments': 13},
 {'title': 'Starting our new Fabric environment - help steer with Fabric Link',
  'text': 'Hi all,\n\nGreat community in here and so good to see Microsoft employees engaging so often with threads.\n\nAnyway, I am basically the project lead on landing our new Fabric environment - for context our main requirement is analytics from D365 F&amp;O but we have quite a range of data sources at the moment so will be good to bring everything into the Fabric capacity. We currently use BYOD into Azure SQL Database. I’m a SQL DBA but more than happy to p

## Step 3: Compare Function Calling with a Standard Chat Model

This step is included only for comparison. It demonstrates the difference between using a function call and asking a standard chat model directly.

In [9]:
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "user", "content": "Summarize the Reddit posts from MicrosoftFabric subreddit posted yesterday."}
    ]
)

print(response.output_text)

I'm sorry, but I can't access or retrieve real-time content from external websites like Reddit. However, if you can provide the text of the posts, I can help summarize them for you.


In [ ]:
# 'output': [ResponseOutputMessage
response.__dict__

{'id': 'resp_02c80578bb486175006a097db3f8cc8196ab1334b0434a175e',
 'created_at': 1779006899.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseOutputMessage(id='msg_02c80578bb486175006a097db5c1788196aa9f7600738e5d6a', content=[ResponseOutputText(annotations=[], text="I'm sorry, but I can't access or retrieve real-time content from external websites like Reddit. However, if you can provide the text of the posts, I can help summarize them for you.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [],
 'top_p': 1.0,
 'background': False,
 'conversation': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'previous_response_id': None,
 'prompt': None,
 'prompt_cache_key': None,
 'prompt_cache_retention': 'in_memory',
 'reasoning': Reasoning

In [11]:
response.output[0].__dict__

{'id': 'msg_02c80578bb486175006a097db5c1788196aa9f7600738e5d6a',
 'content': [ResponseOutputText(annotations=[], text="I'm sorry, but I can't access or retrieve real-time content from external websites like Reddit. However, if you can provide the text of the posts, I can help summarize them for you.", type='output_text', logprobs=[])],
 'role': 'assistant',
 'status': 'completed',
 'type': 'message'}

## Step 4: Define the Input Schema for the Reddit Tool

This step describes the `get_yesterday_reddit_posts` function as a tool, including the input parameters OpenAI can use when deciding whether to call it.

In [15]:
tools = [{
    'type': 'function',
    'name': 'get_yesterday_reddit_posts',
    'description': 'Fetch yesterday\'s Reddit posts from a specific subreddit',
    'parameters': {
        'type': 'object',
        'properties': {
            'subreddit_name': {'type': 'string'},
            'limit': {'type': 'integer', 'default': 100}
        },
        'required': ['subreddit_name', 'limit'],
        'additionalProperties': False
    },
    'strict': True
}]

## Step 5: Pass the tool schema over to the model

In [ ]:
# this is a message we want LLM to process to get the arguments for the tool call
input_messages = [{"role": "user", "content": "Summarize the Reddit posts from MicrosoftFabric subreddit posted yesterday."}]

In [16]:
response = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
)

In [ ]:
# LLM does not provide the text output at this point, it only provides the arguments for the tool call
print(response.output_text)

In [ ]:
#[ResponseFunctionToolCall
response.__dict__

{'id': 'resp_0d029abacfe00c8d006a097fc02d3c81959152bd6377e06e42',
 'created_at': 1779007424.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseFunctionToolCall(arguments='{"subreddit_name":"MicrosoftFabric","limit":100}', call_id='call_I18we0h8mrgEstyFyba3D33l', name='get_yesterday_reddit_posts', type='function_call', id='fc_0d029abacfe00c8d006a097fc1d6048195b1890d15350c947e', status='completed')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='get_yesterday_reddit_posts', parameters={'type': 'object', 'properties': {'subreddit_name': {'type': 'string'}, 'limit': {'type': 'integer', 'default': 100}}, 'required': ['subreddit_name', 'limit'], 'additionalProperties': False}, strict=True, type='function', description="Fetch yesterday's Reddit posts from a specific subreddit")],
 'top_p': 1.0,
 'background': False,
 'c

In [ ]:
# This is a function call to get_yesterday_reddit_posts with the parameters specified 
response.output[0].__dict__

{'arguments': '{"subreddit_name":"MicrosoftFabric","limit":100}',
 'call_id': 'call_I18we0h8mrgEstyFyba3D33l',
 'name': 'get_yesterday_reddit_posts',
 'type': 'function_call',
 'id': 'fc_0d029abacfe00c8d006a097fc1d6048195b1890d15350c947e',
 'status': 'completed'}

## Step 6: Format the tool call response from the LLM

In [20]:
import json

tool_call = response.output[0]
args = json.loads(tool_call.arguments)

In [21]:
print(tool_call)

ResponseFunctionToolCall(arguments='{"subreddit_name":"MicrosoftFabric","limit":100}', call_id='call_I18we0h8mrgEstyFyba3D33l', name='get_yesterday_reddit_posts', type='function_call', id='fc_0d029abacfe00c8d006a097fc1d6048195b1890d15350c947e', status='completed')


In [ ]:
# LLM parsed the request and decided to call the tool with the following arguments
print(args)

{'subreddit_name': 'MicrosoftFabric', 'limit': 100}


## Step 7: Pass on the tool call arguments to our tool/python function

We now need to pass on the arguments received by the model to our python function or tool

In [23]:
result = get_yesterday_reddit_posts(args['subreddit_name'], args['limit'])
result

[{'title': '95 GB data warehouse - Azure SQL DB in Fabric make sense.',
  'text': "We're migrating to Fabric from a current setup that is a SQL Server Data warehouse, largely fed by a Synapse Datalake containing Delta-Parquet, Parquet, and CSV files. \n\nFor the Datawarehouse part, given the relatively low data volume, sticking with a RDMS is appealing vs. Delta Parquet storage.  \n\nThat said, thoughts/ feedback?",
  'score': 8,
  'comments': 13},
 {'title': 'Starting our new Fabric environment - help steer with Fabric Link',
  'text': 'Hi all,\n\nGreat community in here and so good to see Microsoft employees engaging so often with threads.\n\nAnyway, I am basically the project lead on landing our new Fabric environment - for context our main requirement is analytics from D365 F&amp;O but we have quite a range of data sources at the moment so will be good to bring everything into the Fabric capacity. We currently use BYOD into Azure SQL Database. I’m a SQL DBA but more than happy to p

## Step 8: Append the response of the tool into the message list

In [24]:
input_messages.append(tool_call)

input_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

In [ ]:
# Pritty print the messages to see the input messages
from pprint import pprint
pprint(input_messages)

[{'content': 'Summarize the Reddit posts from MicrosoftFabric subreddit posted '
             'yesterday.',
  'role': 'user'},
 ResponseFunctionToolCall(arguments='{"subreddit_name":"MicrosoftFabric","limit":100}', call_id='call_I18we0h8mrgEstyFyba3D33l', name='get_yesterday_reddit_posts', type='function_call', id='fc_0d029abacfe00c8d006a097fc1d6048195b1890d15350c947e', status='completed'),
 {'call_id': 'call_I18we0h8mrgEstyFyba3D33l',
  'output': "[{'title': '95 GB data warehouse - Azure SQL DB in Fabric make "
            'sense.\', \'text\': "We\'re migrating to Fabric from a current '
            'setup that is a SQL Server Data warehouse, largely fed by a '
            'Synapse Datalake containing Delta-Parquet, Parquet, and CSV '
            'files. \\n\\nFor the Datawarehouse part, given the relatively low '
            'data volume, sticking with a RDMS is appealing vs. Delta Parquet '
            'storage.  \\n\\nThat said, thoughts/ feedback?", \'score\': 8, '
            "'c

## Step 9: Pass the message list into the model

In [26]:
response_2 = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
) 

In [ ]:
# Here we expect LLM to provide the summary of the posts retrieved from the subreddit specified in the user message
print(response_2.output_text)

Here is a summary of the Reddit posts from the MicrosoftFabric subreddit posted yesterday:

1. A discussion about the data size and storage options in Fabric, specifically regarding a 95 GB data warehouse and whether to use Azure SQL DB or Delta Parquet storage.
2. A post from a project lead sharing their experience and challenges in setting up a new Fabric environment, especially related to Fabric Link and Dataverse integration.
3. An informational post explaining how to lock files in OneLake using the ADLS Gen2 Lease API, emphasizing its usefulness for mutual exclusion over files.
4. A query about the status of the May update for Power BI Desktop, with concerns about whether it was pulled.
5. A question from a developer about availability of alpha/beta testing environments for new Fabric features, seeking ways to access unreleased versions for testing and feedback.

Would you like a more detailed summary or insights into any specific post?


In [ ]:
#'output': [ResponseOutputMessage
response_2.__dict__

{'id': 'resp_0d029abacfe00c8d006a0980970f3081958159e2da4269c850',
 'created_at': 1779007639.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseOutputMessage(id='msg_0d029abacfe00c8d006a09809755148195b5fb00dc86a268e7', content=[ResponseOutputText(annotations=[], text='Here is a summary of the Reddit posts from the MicrosoftFabric subreddit posted yesterday:\n\n1. A discussion about the data size and storage options in Fabric, specifically regarding a 95 GB data warehouse and whether to use Azure SQL DB or Delta Parquet storage.\n2. A post from a project lead sharing their experience and challenges in setting up a new Fabric environment, especially related to Fabric Link and Dataverse integration.\n3. An informational post explaining how to lock files in OneLake using the ADLS Gen2 Lease API, emphasizing its usefulness for mutual exclusion over files.\n4. A query about the